# 01 — Synthetic data generation

Generates reproducible synthetic instances for the medical duty rostering model.

**Data status: synthetic.** These data do not describe any real hospital. They are calibrated with public information (regulation, Madrid public holiday calendar) and the assumptions listed in Section 2, which must be validated with a hospital service.

- Output: `data/synthetic/<instance_name>/` (CSV files plus `meta.json`).
- Notation: [`docs/formulation.md`](../docs/formulation.md).

## 1. Setup

In [1]:
import json
import math
from datetime import date, timedelta
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

SEED = 2026
DATA_DIR = Path("..") / "data" / "synthetic"

## 2. Assumptions

| Item | Assumption | Formulation symbol |
|---|---|---|
| Service size | 15 residents, 3 per residency year (R1–R5) | $\mathcal{R}$ |
| Seniors | Residents in year R3 or above | $\mathcal{R}_S$ |
| Duty types | `ED` (emergency department, all years) and `WARD` (ward on-call, R2 and above) | $\mathcal{G}$, $E_{rdg}$ |
| Coverage | `ED`: 2 residents per day, at least 1 senior, at most 1 R1. `WARD`: 1 resident per day | $N_{dg}$, $S_{dg}$, $U_{dg}$ |
| Monthly limits | Minimum 4 duties (training), prorated by days available; maximum 7 (RD 1146/2006) | $G^{\min}_r$, $G^{\max}_r$ |
| Ordinary activity | 9 residents needed on every working day | $PL_d$ |
| Demanding days | Saturdays, Sundays and public holidays of the Community of Madrid | $\mathcal{F}$ |
| Availability | External rotations (whole month), vacation blocks, single course days | $E_{rdg}$, $ROT_{rd}$ |
| Requests | 0–3 requested days off per resident (half of them on weekends); priority 1 (50%), 2 (35%), 3 (15%) | $\mathcal{P}$, $Q_{rd}$ |
| Reference period | Residency year starting in June; history of weekend/holiday duties drawn around the fair share with a resident-specific bias, so that the history is unbalanced | $H_r$, $\Phi_r$ |
| Previous month | 3 residents on duty on day $0$ and 3 others on day $-1$ | $Y_{rd}$ |

The configuration of each instance is stored in its `meta.json`.

In [2]:
BASE_CONFIG = {
    "instance_name": "jan2027_base",
    "year": 2027,
    "month": 1,
    "residents_per_year": {1: 3, 2: 3, 3: 3, 4: 3, 5: 3},
    "senior_min_year": 3,
    "duty_types": {
        "ED": {"name": "Emergency department", "min_year": 1, "N": 2, "S": 1, "U": 1},
        "WARD": {"name": "Ward on-call", "min_year": 2, "N": 1, "S": 0, "U": 0},
    },
    "G_min": 4,
    "G_max": 7,
    "PL_working_day": 9,
    "external_rotations": 1,
    "vacation_blocks": 2,
    "vacation_length": 5,
    "course_days": 4,
    "max_requests_per_resident": 3,
    "request_priorities": {1: 0.50, 2: 0.35, 3: 0.15},
    "reference_start": [2026, 6],
    "history_bias_sd": 0.30,
    "previous_month_on_duty": 3,
}

DEC_CONFIG = {
    **BASE_CONFIG,
    "instance_name": "dec2026_christmas",
    "year": 2026,
    "month": 12,
    "vacation_blocks": 6,
    "vacation_length": 7,
}

## 3. Generator

Each block below builds the parameters of one part of the formulation. All randomness comes from a single `numpy` generator seeded with `SEED`, so every instance is reproducible.

In [3]:
def build_calendar(cfg):
    # Days D = {1..n}, demanding days F (weekends + Madrid public holidays), working days L.
    y, m = cfg["year"], cfg["month"]
    n = (date(y + m // 12, m % 12 + 1, 1) - date(y, m, 1)).days
    madrid = holidays.Spain(subdiv="MD", years=y)
    rows = []
    for d in range(1, n + 1):
        day = date(y, m, d)
        is_holiday = day in madrid
        is_F = day.weekday() >= 5 or is_holiday
        rows.append({"day": d, "date": day.isoformat(), "weekday": day.strftime("%a"),
                     "is_holiday": is_holiday, "is_F": is_F,
                     "PL": 0 if is_F else cfg["PL_working_day"]})
    return pd.DataFrame(rows)


def build_residents(cfg):
    rows = []
    for year, count in cfg["residents_per_year"].items():
        for _ in range(count):
            rows.append({"year": year})
    res = pd.DataFrame(rows)
    res.insert(0, "resident", [f"R{i:02d}" for i in range(1, len(res) + 1)])
    res["senior"] = res["year"] >= cfg["senior_min_year"]
    res["first_year"] = res["year"] == 1
    return res


def build_availability(cfg, res, cal, rng):
    # Single availability calendar from which E_rdg and ROT_rd are derived.
    n = len(cal)
    avail = pd.DataFrame(True, index=res["resident"], columns=cal["day"])
    reason = pd.DataFrame("", index=res["resident"], columns=cal["day"])
    # External rotations: whole month away (R3-R4 residents, typical rotation years).
    candidates = res.loc[res["year"].isin([3, 4]), "resident"].to_numpy()
    away = rng.choice(candidates, size=cfg["external_rotations"], replace=False)
    avail.loc[away, :] = False
    reason.loc[away, :] = "external_rotation"
    # Vacation blocks.
    present = [r for r in res["resident"] if r not in away]
    for r in rng.choice(present, size=cfg["vacation_blocks"], replace=False):
        start = int(rng.integers(1, n - cfg["vacation_length"] + 2))
        days = list(range(start, start + cfg["vacation_length"]))
        avail.loc[r, days] = False
        reason.loc[r, days] = "vacation"
    # Single course days.
    for _ in range(cfg["course_days"]):
        r, d = rng.choice(present), int(rng.integers(1, n + 1))
        if avail.loc[r, d]:
            avail.loc[r, d] = False
            reason.loc[r, d] = "course"
    return avail, reason, list(away)


def build_demand(cfg, cal):
    rows = []
    for d in cal["day"]:
        for g, spec in cfg["duty_types"].items():
            rows.append({"day": d, "duty": g, "N": spec["N"], "S": spec["S"], "U": spec["U"]})
    return pd.DataFrame(rows)


def build_eligibility(cfg, res, avail):
    # E_rdg = 1 if qualified for duty type g (residency year) and available on day d.
    rows = []
    for _, row in res.iterrows():
        for d in avail.columns:
            for g, spec in cfg["duty_types"].items():
                rows.append({"resident": row["resident"], "day": d, "duty": g,
                             "E": int(avail.loc[row["resident"], d] and row["year"] >= spec["min_year"])})
    return pd.DataFrame(rows)


def build_monthly_limits(cfg, res, avail):
    days_avail = avail.sum(axis=1)
    n = avail.shape[1]
    res = res.copy()
    res["days_available"] = res["resident"].map(days_avail).astype(int)
    res["G_min"] = [math.floor(cfg["G_min"] * a / n) for a in res["days_available"]]
    res["G_max"] = [cfg["G_max"] if a > 0 else 0 for a in res["days_available"]]
    return res


def build_requests(cfg, res, cal, avail, rng):
    probs = cfg["request_priorities"]
    weekend = cal.loc[cal["is_F"], "day"].to_numpy()
    rows = []
    for r in res["resident"]:
        free = [d for d in cal["day"] if avail.loc[r, d]]
        if not free:
            continue
        k = int(rng.integers(0, cfg["max_requests_per_resident"] + 1))
        chosen = set()
        for _ in range(k):
            pool = [d for d in (weekend if rng.random() < 0.5 else free) if d in free and d not in chosen]
            if pool:
                chosen.add(int(rng.choice(pool)))
        for d in sorted(chosen):
            q = int(rng.choice(list(probs), p=list(probs.values())))
            rows.append({"resident": r, "day": d, "Q": q})
    return pd.DataFrame(rows, columns=["resident", "day", "Q"])


def build_history(cfg, res, cal, avail, rng):
    # H_r: weekend/holiday duties in previous months of the reference period.
    # Phi_r: fraction of the reference period (including this month) available.
    y0, m0 = cfg["reference_start"]
    months_before = (cfg["year"] - y0) * 12 + (cfg["month"] - m0)
    f_positions = cal["is_F"].sum() * sum(s["N"] for s in cfg["duty_types"].values())
    fair_share = f_positions / len(res)  # weekend/holiday duties per resident per month
    res = res.copy()
    H, phi = [], []
    for r, current in zip(res["resident"], res["days_available"] / len(cal)):
        past = rng.choice([1.0, 1.0, 1.0, 1.0, 0.0, 0.5], size=months_before)  # rotations in the past
        bias = rng.lognormal(0.0, cfg["history_bias_sd"])
        H.append(int(rng.poisson(fair_share * past.sum() * bias)))
        phi.append(round((past.sum() + current) / (months_before + 1), 4))
    res["H"] = H
    res["Phi"] = phi
    return res


def build_previous_month(cfg, res, avail, rng):
    present = [r for r in res["resident"] if avail.loc[r].any()]
    k = cfg["previous_month_on_duty"]
    picked = rng.choice(present, size=2 * k, replace=False)
    rows = [{"resident": r, "day": 0, "on_duty": 1} for r in picked[:k]]
    rows += [{"resident": r, "day": -1, "on_duty": 1} for r in picked[k:]]
    return pd.DataFrame(rows)


def generate_instance(cfg, seed):
    rng = np.random.default_rng(seed)
    cal = build_calendar(cfg)
    res = build_residents(cfg)
    avail, reason, away = build_availability(cfg, res, cal, rng)
    res = build_monthly_limits(cfg, res, avail)
    res = build_history(cfg, res, cal, avail, rng)
    rot = avail.copy()
    rot.loc[:, cal.loc[cal["is_F"], "day"].to_list()] = False  # ROT_rd only on working days
    inst = {
        "config": cfg,
        "calendar": cal,
        "residents": res,
        "demand": build_demand(cfg, cal),
        "eligibility": build_eligibility(cfg, res, avail),
        "availability": (avail.astype(int).stack().rename("available").reset_index()
                         .merge(rot.astype(int).stack().rename("ROT").reset_index())
                         .merge(reason.stack().rename("reason").reset_index())),
        "requests": build_requests(cfg, res, cal, avail, rng),
        "previous_month": build_previous_month(cfg, res, avail, rng),
    }
    inst["availability"].columns = ["resident", "day", "available", "ROT", "reason"]
    return inst

## 4. Validation checks

Domain and consistency checks on every parameter, plus an aggregate capacity report. The capacity report is informative only: the model never becomes infeasible because unmet requirements are measured by shortfall variables.

In [4]:
def validate_instance(inst):
    res, cal, dem = inst["residents"], inst["calendar"], inst["demand"]
    E, av, req, prev = inst["eligibility"], inst["availability"], inst["requests"], inst["previous_month"]
    assert (dem[["N", "S", "U"]] >= 0).all().all() and (dem["S"] <= dem["N"]).all()
    assert E["E"].isin([0, 1]).all() and av["ROT"].isin([0, 1]).all()
    assert res["Phi"].between(0, 1).all() and (res["H"] >= 0).all()
    assert (res["G_min"] <= res["G_max"]).all()
    assert (req["Q"] > 0).all() and not req.duplicated(["resident", "day"]).any()
    # E and ROT come from the same availability calendar.
    merged = E.merge(av, on=["resident", "day"])
    assert not ((merged["E"] == 1) & (merged["available"] == 0)).any()
    assert not ((av["ROT"] == 1) & (av["available"] == 0)).any()
    # Previous month: nobody on duty on two consecutive days (-1 and 0).
    assert not prev["resident"].duplicated().any()
    return True


def capacity_report(inst):
    res, dem, E = inst["residents"], inst["demand"], inst["eligibility"]
    cal = inst["calendar"]
    seniors = set(res.loc[res["senior"], "resident"])
    present = E.groupby(["resident", "day"])["E"].max().groupby("day").sum()
    return pd.Series({
        "days": len(cal),
        "weekend/holiday days": int(cal["is_F"].sum()),
        "duty positions (sum N)": int(dem["N"].sum()),
        "senior positions (sum S)": int(dem["S"].sum()),
        "capacity (sum G_max)": int(res["G_max"].sum()),
        "training minima (sum G_min)": int(res["G_min"].sum()),
        "senior capacity (sum G_max seniors)": int(res.loc[res["resident"].isin(seniors), "G_max"].sum()),
        "min residents available on a day": int(present.min()),
        "requests": len(inst["requests"]),
    })

## 5. Generate the instances

In [5]:
instances = {cfg["instance_name"]: generate_instance(cfg, SEED) for cfg in (BASE_CONFIG, DEC_CONFIG)}
for name, inst in instances.items():
    validate_instance(inst)
pd.DataFrame({name: capacity_report(inst) for name, inst in instances.items()})

,jan2027_base,dec2026_christmas
days,31,31
weekend/holiday days,12,11
duty positions (sum N),93,93
senior positions (sum S),31,31
capacity (sum G_max),98,98
training minima (sum G_min),51,46
senior capacity (sum G_max seniors),56,56
min residents available on a day,11,9
requests,17,12


`jan2027_base` is the baseline month. `dec2026_christmas` is a demanding month (three public holidays and more vacations), used to test the model when capacity is tight.

In [6]:
base = instances["jan2027_base"]
base["residents"]

,resident,year,senior,first_year,days_available,G_min,G_max,H,Phi
0,R01,1,False,True,26,3,7,17,0.7923
1,R02,1,False,True,30,3,7,13,0.8085
2,R03,1,False,True,26,3,7,14,0.7923
3,R04,2,False,False,31,4,7,16,1.0000
4,R05,2,False,False,31,4,7,22,0.8125
5,R06,2,False,False,31,4,7,9,0.8750
6,R07,3,True,False,31,4,7,26,0.8125
7,R08,3,True,False,31,4,7,14,0.6875
8,R09,3,True,False,31,4,7,6,0.5625
9,R10,4,True,False,29,3,7,8,0.6169


In [7]:
avail_matrix = base["availability"].pivot(index="resident", columns="day", values="available")
avail_matrix.style.map(lambda v: "background-color: #f4cccc" if v == 0 else "")

day,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31
resident,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
R01,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1
R02,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
R03,1,1,1,1,1,1,1,1,1,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
R04,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
R05,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
R06,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
R07,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
R08,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
R09,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


## 6. Export

In [8]:
def export_instance(inst, seed, root=DATA_DIR):
    out = root / inst["config"]["instance_name"]
    out.mkdir(parents=True, exist_ok=True)
    for key in ["calendar", "residents", "demand", "eligibility", "availability", "requests", "previous_month"]:
        inst[key].to_csv(out / f"{key}.csv", index=False)
    meta = {"data_status": "synthetic", "seed": seed, "generator": "notebooks/01_synthetic_data_generation.ipynb",
            "config": inst["config"]}
    (out / "meta.json").write_text(json.dumps(meta, indent=2, default=str))
    return out


for inst in instances.values():
    print("exported", export_instance(inst, SEED))

exported ../data/synthetic/jan2027_base
exported ../data/synthetic/dec2026_christmas
